In [0]:
from pyspark.sql import functions as f
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid


In [0]:
spark.sql("use catalog novacart_adb")
spark.sql("create schema if not exists silver_schema")

silver_run_id = str(uuid.uuid4())
print("current silver run id", silver_run_id)


In [0]:
spark.sql("""
          create table if not exists novacart_adb.silver_schema.processing_control (
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingested_at timestamp,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp

          )
           using delta 
          """
)

Helper Functions

In [0]:
def upsert_to_silver(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark,target_table)
        (dt.alias("target")
        .merge(df_source.alias("source"),f"target.{join_key} = source.{join_key}")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    else:
      df_source.write.format("delta").saveAsTable(target_table)
      

In [0]:
def get_last_processed_bronze_ingested_at(entity_name:str):
    ctrl = (
            spark.table("novacart_adb.silver_schema.processing_control")
            .filter(
                (f.col("layer") == "silver") &
                (f.col("entity_name") == entity_name) &
                (f.col("run_status") == "success")
            )
            .orderBy(f.col("updated_at").desc())
            .limit(1)
    )
    
    
    rows = ctrl.collect()
    if not rows:
        return None
  
    return rows[0]["last_processed_bronze_ingested_at"]
,
    

In [0]:
def upsert_silver_control(entity_name,last_processed_bronze_run_id,last_processed_bronze_ingested_at,rows_merged):
    ctrl_df = spark.createDataFrame(
        [
            (
                "silver",
                entity_name,
                last_processed_bronze_run_id,
                last_processed_bronze_ingested_at,
                int(rows_merged),
                "success",
                silver_run_id,
                datetime.now()
            )],
        schema = """
        layer string, 
        entity_name string, 
        last_processed_bronze_run_id string, 
        last_processed_bronze_ingested_at timestamp, 
        rows_merged bigint, 
        run_status string, 
        silver_run_id string,
        updated_at timestamp
        """
        )
    
    dt = DeltaTable.forName(spark,"novacart_adb.silver_schema.processing_control")
    (dt.alias("t")
        .merge(ctrl_df.alias("s"),f"t.entity_name = s.entity_name")
        .whenMatchedUpdate(set={
            "last_processed_bronze_run_id" : "s.last_processed_bronze_run_id",
            "last_processed_bronze_ingested_at" : "s.last_processed_bronze_ingested_at",
            "rows_merged" : "s.rows_merged",
            "run_status" : "s.run_status",
            "silver_run_id" : "s.silver_run_id",
            "updated_at" : "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute())


In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    last_ingested_at = get_last_processed_bronze_ingested_at(entity_name)
    bronze_df = spark.read.table(bronze_table)
    if last_ingested_at is None:
        return bronze_df,last_ingested_at
    
        return bronze_df.filter(f.col("bronze_ingested_at") > f.lit(last_ingested_at)),last_ingested_at

In [0]:
df_raw = spark.sql("select * from novacart_adb.bronze_schema.orders_raw")
display(df_raw)

In [0]:
orders_inc,last_orders_ingested_at = get_incremental_bronze("novacart_adb.bronze_schema.orders_raw","orders")

orders_inc_count = orders_inc.count()
print(f"Orders rows to process in silver layer: {orders_inc_count}")

if orders_inc_count > 0:
    order_window = Window.partitionBy("order_id").orderBy(
        f.col("updated_at").cast("timestamp").desc(),
        f.col("bronze_ingested_at").desc()
    )

    orders_cleaned = (
        orders_inc
        .withColumn("order_status", f.upper(f.trim(f.col("order_status"))))
        .withColumn("order_status", f.when(f.col("order_status") == "", f.lit(None)).otherwise(f.col("order_status")))
        .withColumn("order_amount", f.regexp_replace(f.col("order_amount"),r"[$, ]",""))
        .withColumn("order_amount", f.when(f.trim(f.col("order_amount")).isin("N/A","NULL","??",""),None).otherwise(f.col("order_amount")))
        .withColumn("order_amount", f.col("order_amount").cast("double"))
        .withColumn("created_at",f.to_timestamp("created_at"))
        .withColumn("row_rank", f.row_number().over(order_window))
        .filter(f.col("row_rank") == 1)
        .drop("row_rank")
        .dropDuplicates()
        .withColumn("silver_run_id",f.lit(silver_run_id)))


    upsert_to_silver(orders_cleaned,"novacart_adb.silver_schema.orders", "order_id")

    orders_validated = (
        orders_cleaned
        .withColumn("to_be_verified_by_orders_team",
                f.when(f.col("customer_id").isNull(),"verify_customer_id")
                 .when(f.col("product_id").isNull(),"verify_product_id")
                .when(f.col("order_status").isNull() | (f.trim(f.col("order_status")) == ""),  "verify_order_status")
                .when((f.col("order_amount").isNull()) | (f.col("order_amount") <= 0),      "verify_order_amount")
                .otherwise("No Issues"))
        .withColumn(
            "check_order_amount",
            f.when((f.col("order_amount").isNotNull()) | (f.col("order_amount") <= 0), f.lit  (True)).otherwise(f.lit(False)))
        .withColumn("order_date", f.to_date("created_at"))
        .withColumn("order_month",f.month("order_date"))
        .withColumn("order_year",f.year("order_date"))
        .withColumn("order_day",f.dayofmonth("order_date"))
        .withColumn("order_dow",f.date_format("created_at","E"))
    )

    orders_good = orders_validated.filter(f.col("to_be_verified_by_orders_team") == "No Issues")
    orders_bad = (
        orders_validated
        .filter(f.col("to_be_verified_by_orders_team") != "No Issues")
        .withColumn("quarantine_ts",f.current_timestamp()))

    upsert_to_silver(orders_good, "novacart_adb.silver_schema.orders_transformed","order_id")

    orders_bad.write.format("delta").mode("append").saveAsTable("novacart_adb.silver_schema.orders_quarantine")

    mx_ingested = orders_inc.agg(f.max(f.col("bronze_ingested_at")).alias("mx")).collect()[0]["mx"]
    mx_run = (
        orders_inc.filter(f.col("bronze_ingested_at") == f.lit(mx_ingested))
        .agg(f.max(f.col("bronze_run_id")).alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control("orders",mx_run,mx_ingested,orders_good.count())
    
else:
    print("No new orders Bronze rows to process in Silver")

    upsert_silver_control(
        "orders",
        None,
        last_orders_ingested_at,
        orders_inc_count
    )



In [0]:
%sql
select * from novacart_adb.silver_schema.orders;


In [0]:
%sql
select * from novacart_adb.silver_schema.orders_quarantine;

In [0]:
%sql
select * from novacart_adb.bronze_schema.products_raw

In [0]:
products_inc,last_orders_ingested_at = get_incremental_bronze("novacart_adb.bronze_schema.products_raw","products")

products_inc_count = products_inc.count()
print(f"Products rows to process in silver layer: {products_inc_count}")

if products_inc_count > 0:
    product_window = Window.partitionBy("product_id").orderBy(
        f.col("updated_at").cast("timestamp").desc(),
        f.col("bronze_ingested_at").desc()
    )

    products_cleaned = (
        products_inc
        .withColumn("product_name", f.upper(f.trim(f.col("product_name"))))
        .withColumn("product_name", f.when(f.col("product_name") == "", f.lit(None)).otherwise(f.col("product_name")))
        .withColumn("category", f.when(f.upper(f.trim(f.col("category"))).contains("ELECTRNICS"),"ELECTRONICS").otherwise(f.upper(f.trim(f.col("category")))))
        .withColumn("price", f.trim(f.col("price")))
        .withColumn("price",f.regexp_replace(f.col("price"),r"\$ ",""))
        .withColumn("price",f.expr("try_cast(price as double)"))
        .withColumn("updated_at",f.to_timestamp(f.col("updated_at")))
        .withColumn("row_rank", f.row_number().over(product_window))
        .filter(f.col("row_rank") == 1)
        .drop("row_rank")
        .dropDuplicates()
        .withColumn("silver_run_id",f.lit(silver_run_id)))


    upsert_to_silver(products_cleaned,"novacart_adb.silver_schema.products_cleaned", "product_id")

    products_validated = (
        products_cleaned
        .withColumn("to_be_verified_by_products_team",
                f.when(f.col("product_name").isNull(),"verify_product_name")
                 .when(f.col("category").isNull(),"verify_category")
                .when((f.col("price").isNull()) | (f.col("price") <= 0),      "verify_price")
                .otherwise("No Issues"))
        .withColumn(
            "check_product_price",
            f.when((f.col("price").isNull()) | (f.col("price") <= 0), "invalid_price").otherwise("valid_price"))
        
    )

    products_good = products_validated.filter(f.col("to_be_verified_by_products_team") == "No Issues")
    if "price_raw" in products_good.columns:
        products_good = products_good.drop("price_raw")

    products_bad = (
        products_validated
        .filter((f.col("to_be_verified_by_products_team") != "No Issues") | (f.col("check_product_price") == "invalid_price"))
        .withColumn("quarantine_ts",f.current_timestamp()))

    upsert_to_silver(products_good, "novacart_adb.silver_schema.products_transformed","product_id")

    products_bad.write.format("delta").mode("append").saveAsTable("novacart_adb.silver_schema.products_quarantine")

    mx_ingested = products_inc.agg(f.max(f.col("bronze_ingested_at")).alias("mx")).collect()[0]["mx"]
    mx_run = (
        products_inc.filter(f.col("bronze_ingested_at") == f.lit(mx_ingested))
        .agg(f.max(f.col("bronze_run_id")).alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control("products",mx_run,mx_ingested,products_good.count())
    
else:
    print("No new products Bronze rows to process in Silver")

    upsert_silver_control(
        "products",
        None,
        last_products_ingested_at,
        products_inc_count
    )



In [0]:
%sql
select * from novacart_adb.silver_schema.products_quarantine

In [0]:
payments_inc,last_payments_ingested_at = get_incremental_bronze("novacart_adb.bronze_schema.payments_raw","payments")

payments_inc_count = payments_inc.count()
print(f"Payments rows to process in silver layer: {payments_inc_count}")

if payments_inc_count > 0:
    payments_window = Window.partitionBy("payment_id").orderBy(
        f.col("processed_at").cast("timestamp").desc(),
        f.col("bronze_ingested_at").desc()
    )

    payments_cleaned = (
        payments_inc
        .withColumn("payment_status", f.upper(f.trim(f.col("payment_status"))))
        .withColumn("payment_status", f.when(f.col("payment_status") == "", f.lit(None)).otherwise(f.col("payment_status")))
        .withColumn("paid_amount", f.trim(f.col("paid_amount")))
        .withColumn("paid_amount", f.regexp_replace(f.col("paid_amount"),r"\$",""))
        .withColumn("paid_amount", f.regexp_replace(f.col("paid_amount"),",","."))
        .withColumn("paid_amount", f.regexp_replace(f.col("paid_amount"),r"s+",""))
        .withColumn("paid_amount", f.expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at",f.to_timestamp("processed_at"))
        .withColumn("row_rank", f.row_number().over(payments_window))
        .filter(f.col("row_rank") == 1)
        .drop("row_rank")
        .dropDuplicates()
        .withColumn("silver_run_id",f.lit(silver_run_id)))


    upsert_to_silver(payments_cleaned,"novacart_adb.silver_schema.payments_cleaned", "payment_id")

    payments_validated = (
        payments_cleaned
        .withColumn("to_be_verified_by_payments_team",
                f.when(f.col("order_id").isNull(),"verify_order_id")
                .when(f.col("payment_status").isNull(), "verify_payment_status")
                .when((f.col("paid_amount").isNull()) | (f.col("paid_amount") <= 0),      "verify_paid_amount")
                .otherwise("No Issues"))
        .withColumn(
            "check_paid_amount",
            f.when((f.col("paid_amount").isNull()) | (f.col("paid_amount") <= 0), f.lit  (True)).otherwise(f.lit(False)))
        
    )

    payments_good = payments_validated.filter(f.col("to_be_verified_by_payments_team") == "No Issues")
    payments_bad = (
        payments_validated
        .filter(f.col("to_be_verified_by_payments_team") != "No Issues")
        .withColumn("quarantine_ts",f.current_timestamp()))

    upsert_to_silver(payments_good, "novacart_adb.silver_schema.payments_transformed","payment_id")

    payments_bad.write.format("delta").mode("append").saveAsTable("novacart_adb.silver_schema.payments_quarantine")

    mx_ingested = payments_inc.agg(f.max(f.col("bronze_ingested_at")).alias("mx")).collect()[0]["mx"]
    mx_run = (
        payments_inc.filter(f.col("bronze_ingested_at") == f.lit(mx_ingested))
        .agg(f.max(f.col("bronze_run_id")).alias("mx"))
        .collect()[0]["mx"]
    )

    upsert_silver_control("payments",mx_run,mx_ingested,payments_good.count())
    
else:
    print("No new payments Bronze rows to process in Silver")

    upsert_silver_control(
        "payments",
        None,
        last_payments_ingested_at,
        payments_inc_count
    )




In [0]:
%sql
select * from novacart_adb.silver_schema.payments_quarantine

In [0]:
print("products transformed count... :", spark.sql("select count(*) from novacart_adb.silver_schema.products_transformed").collect()[0][0])
print("orders transformed count... :", spark.sql("select count(*) from novacart_adb.silver_schema.orders_transformed").collect()[0][0])
print("payments transformed count... :", spark.sql("select count(*) from novacart_adb.silver_schema.payments_transformed").collect()[0][0])
display(spark.table("novacart_adb.silver_schema.processing_control").orderBy("entity_name"))